# FashionLeNet: software study and Colab CPU baseline

This notebook runs the same checked Python entry points as the repository. Included result plots/checkpoints come from the documented release run. Notebook cells themselves are not pre-executed. Use the full official test set for reported accuracy. FPGA metrics still require Vivado and an actual target.

## 1. Open the repository

Locally, launch this notebook from the extracted repository. In Colab, upload the supplied complete GitHub ZIP when prompted. This keeps all source files, models and fixtures together.

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
if not Path("pyproject.toml").exists():
    candidates = [Path.cwd().parent, Path("/content/FashionLeNet")]
    found = next((p for p in candidates if (p/"pyproject.toml").exists()), None)
    if found is None:
        from google.colab import files
        uploaded = files.upload()
        archive = next(name for name in uploaded if name.endswith(".zip"))
        with zipfile.ZipFile(archive) as z:
            base = Path("/content").resolve()
            for item in z.infolist():
                if not (base/item.filename).resolve().is_relative_to(base):
                    raise ValueError("Unsafe archive path")
            z.extractall(base)
        found = Path("/content/FashionLeNet")
    os.chdir(found)
print(Path.cwd())
assert Path("pyproject.toml").exists()
subprocess.run([sys.executable,"-m","pip","install","-e",".[train]"], check=True)


## 2. Download and verify the official dataset

The loader verifies gzip file checksums. Training images are kept separate from the 10,000-image test set. Native 28x28 and zero-padded 32x32 models are distinct.

In [ ]:
def run(*args):
    subprocess.run([sys.executable,*args], check=True)
def cli(*args):
    run("-m","fashionlenet.cli",*args)
cli("download","--data","data")


## 3. Train or inspect the provided baseline

Set `RETRAIN=True` to perform both documented full training runs. Otherwise the included selected checkpoints are used. The saved best checkpoint is selected on 5,000 validation images, never on test accuracy.

In [ ]:
RETRAIN = False
if RETRAIN:
    cli("train","--input-size","32","--epochs","12","--output","results/trained32")
    cli("train","--input-size","28","--epochs","15","--output","results/trained28")
import json
print(json.loads(Path("results/trained32/training_manifest.json").read_text()))
from IPython.display import display, SVG
display(SVG(filename="docs/figures/training.svg"))


## 4. Quantization and full test-set accuracy

This uses 500 training calibration images, signed INT8 operands, INT32 biases/accumulators, and exact documented shift rounding. PTQ itself performs no retraining.

In [ ]:
cli("quantize","--checkpoint","results/trained32/best.pt","--samples","500","--output","runs/colab_int8.npz")
cli("evaluate","--model","results/trained32/best.pt","--output","runs/colab_fp32_test.json")
cli("evaluate","--model","runs/colab_int8.npz","--output","runs/colab_int8_test.json")
print(json.loads(Path("runs/colab_int8_test.json").read_text())["accuracy"])


## 5. All prescribed pruning settings

Set `RERUN_PRUNING=True` for both methods at 25/50/70/90/95/97/99%. The default displays real stored release results. Structural pruning physically removes channels; unstructured zeroing preserves dense storage.

In [ ]:
RERUN_PRUNING = False
if RERUN_PRUNING:
    cli("prune","--checkpoint","results/trained28/best.pt","--output","runs/colab_pruning28")
    cli("prune","--checkpoint","results/trained28/best.pt","--output","runs/colab_recovery28","--fractions","0.50","--finetune-epochs","3")
import pandas as pd
display(pd.read_csv("results/pruning28/summary.csv"))
display(SVG(filename="docs/figures/pruning.svg"))
display(SVG(filename="docs/figures/recovery.svg"))


## 6. Measure the required Colab CPU baseline

This actually runs on the notebook host CPU, batch one and one PyTorch thread, with warm-up and latency percentiles. It excludes image preprocessing and cannot establish FPGA speedup without an achieved FPGA clock.

In [ ]:
cli("benchmark","--checkpoint","results/trained32/best.pt","--threads","1","--repeats","200","--output","runs/colab_cpu.json")
cpu = json.loads(Path("runs/colab_cpu.json").read_text())
print({k:v for k,v in cpu.items() if k != "samples_ms"})


## 7. Export a real image and run integer inference

The export contains host-loadable memory, model header, label/provenance and every INT32/INT8 golden stage. It uses no hidden floating-point inference operation.

In [ ]:
cli("export","--model","runs/colab_int8.npz","--index","0","--output","build/colab_image")
from fashionlenet.quant import load
from fashionlenet.reference import infer
from fashionlenet.data import load as load_data, preprocess
model=load("runs/colab_int8.npz")
images,labels=load_data("data",False)
pred,logits=infer(model,preprocess(images[:1],32))
print("Predicted", int(pred[0]), "true", int(labels[0]), "logits", logits.tolist())


## 8. Optional Linux RTL execution

Install Icarus only if you want RTL simulation in Colab. This is a logic simulation, not a physical FPGA run. The full top checks every stage and signed accumulator result.

In [ ]:
RUN_RTL = False
if RUN_RTL:
    subprocess.run(["apt-get","update"],check=True)
    subprocess.run(["apt-get","install","-y","iverilog"],check=True)
    run("scripts/run_rtl.py")
    run("scripts/run_rtl.py","--full","--bundle","build/colab_image")


## 9. Analyze clocks and preserve your measurements

The transaction model counts the actual byte-port/cache protocol. Converting clocks using 100 MHz below is a conditional estimate, not an achieved frequency. Download `runs/colab_cpu.json` and the evaluation JSON for your report. Follow `fpga/README.md` for mapped resources, routing and power.

In [ ]:
from fashionlenet.sim import simulate_bundle
modeled=simulate_bundle("build/colab_image",cache=True,latency=4,sparse=True)
print({k:v for k,v in modeled.items() if k != "phases"})
print("Conditional latency at 100 MHz (ms):", modeled["cycles"]/100_000)
try:
    from google.colab import files
    files.download("runs/colab_cpu.json")
except ImportError:
    print("CPU report saved at",Path("runs/colab_cpu.json").resolve())
